<h2>Before you start</h2>
If this is the first time the pipeline is running on this machine, just run the cell below. It will copy startup.py from the BEARMIND folder into your local startup folder. This allows the code in startup.py to be executed automatically after each kernel restart (and removes the need to monotonously click through all setup cells after each reloading).

In [2]:
import shutil
import os

local_startup_dir = get_ipython().profile_dir.startup_dir
filedir = os.getcwd()
shutil.copy(os.path.join(filedir, 'startup.py'), os.path.join(local_startup_dir, 'startup.py'))

'/Users/nikita/.ipython/profile_default/startup/startup.py'

<h2>Module 0</h2>
You need to specify the root folder and pathway pattern. Note that * is a wildcard for any symbol combination except slashes (i.e., for any folder name), so it is strongly recommended to use it here.<br/><br/>
NB!! Just in case, use double backslashes for folder separation, otherwise some symbols may be interpreted as escape sequences. 

In [ ]:
config_data = {
    'ROOT': "C:\\Users\\admin\\YandexDisk\\_Projects\\FOF\\CalciumData\\4_Estimates\\",
    #'DATA_PATHWAY': 'legacy',
    'DATA_PATHWAY': 'bonsai',
    'TEMP_PATHWAY': "c:\\Users\\1\\caiman_data\\temp\\"
}

update_config(config_data)

In [ ]:
CONFIG

In [ ]:
create_mouse_configs(root=CONFIG['ROOT'])
create_session_configs(root=CONFIG['ROOT'])

<h2>Module 1</h2>
Manual video inspection. <br/>Open folder with miniscopic videos in a pop-up window, wait for loading and specify margins to be cropped by sliders or by keyboard, then save them by running the next cell. At the time, cropping .pickle files are to be created in these folders. Repeat for all folders with miniscopic videos you would like to analyze.   

In [ ]:
#Manual file selection:
fnames = list(askopenfilenames(title = 'Select files for inspection', initialdir = CONFIG['ROOT'], filetypes = [('AVI files', '.avi')]))

data = LoadSelectedVideos(fnames)
w = DrawCropper(data, fname=fnames[0])

Batch cropping and timestamp extraction.<br/>Miniscopic videos from folders with .pickle files are to be cropped and saved as _CR.tif in the root folder. There is no need for renaming of sigle-digit .avi files (like 0-9.avi to 00-09.avi)!<br/>
Also, along with video data, timestamps are to be copied from minicopic folders to the root folder. Do not delete them, they are nessesary for the further steps!

## Combined Modules 1-2-2.5

In [ ]:
#Batch crop
cpath_template = os.path.normpath(os.path.join(CONFIG['ROOT'], folder_structure, '*cropping.pickle'))
pick_names = glob(cpath_template)

print([get_session_name_from_path(fname) for fname in pick_names])
# TODO: read from mouse or sconfig only!

for name in pick_names:
    DoCropAndRewrite(name, sort = True, write_mp4 = True)
    extract_and_copy_ts(name)

#Automatic file selection
fnames = glob(os.path.join(CONFIG['ROOT'], '*_CR.tif'))
#OR, alternatively, you can use manual file selection:
#fnames = askopenfilenames(title = 'Select files for motion correction', initialdir = CONFIG['ROOT'], filetypes = [('TIFF files', '.tif')])

mc_dict = {
    'pw_rigid': False,         # flag for performing piecewise-rigid motion correction (otherwise just rigid)
    'max_shifts': (35, 35),    # maximum allowed rigid shift
    'gSig_filt': (8, 8),       # size of high pass spatial filtering, used in 1p data
    'strides': (48, 48),       # start a new patch for pw-rigid motion correction every x pixels
    'overlaps': (24, 24),      # overlap between pathes (size of patch strides+overlaps)
    'max_deviation_rigid': 15,  # maximum deviation allowed for patch with respect to rigid shifts
    'border_nan': 'copy',      # replicate values along the boundaries
    'use_cuda': True,          # Set to True in order to use GPU
    'memory_fact': CONFIG['RAM']/16.0,          # How much memory to allocate. 1 works for 16Gb, so 0.8 showd be optimized for 12Gb.
    'niter_rig': 1,
    'splits_rig': 20,          # for parallelization split the movies in  num_splits chuncks across time
                               # if none all the splits are processed and the movie is saved
    'num_splits_to_process_rig': None,
    'write_mp4': True} # intervals at which patches are laid out for motion correction  

for name in tqdm.tqdm(fnames):
    DoMotionCorrection(name, mc_dict)
    temp_pathway = CONFIG['TEMP_PATHWAY']
    sub_name = '/'.join(name.split('/')[6:])
    memmap_name = f'{temp_pathway}/{sub_name}'
    CleanMemmaps(memmap_name)

#%matplotlib ipympl
fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))
#fnames = askopenfilenames(title = 'Select files for corr image testing',
#                          initialdir = CONFIG['ROOT'],
#                          filetypes = [('TIFF files', '.tif')])

plot_gsig_range(fnames, maxframes=10000, min_gsig=3, max_gsig=6, step=5, dpi=300,
                    show_images=0, save_images=1)
plot_min_corr_and_pnr_range(fnames, maxframes=2000,
                            gsig_range=[3,4,5], pnr_range=[5,7,10],
                            mincorr_range=[0.85, 0.9, 0.95],
                            step=5, dpi=300,
                            show_images=0, save_images=1)

<h2>Module 2</h2>
Batch motion correction.<br/>All _CR.tif files in the root folder are to be automatically motion corrected with NoRMCorre routine [Pnevmatikakis, Giovanucci, 2017] with the parameters below and saved as _MC.tif files.

In [ ]:
#Automatic file selection
#fnames = glob(os.path.join(CONFIG['ROOT'], '*_CR.tif'))
#OR, alternatively, you can use manual file selection:
fnames = askopenfilenames(title = 'Select files for motion correction', initialdir = CONFIG['ROOT'], filetypes = [('TIFF files', '.tif')])

mc_dict = {
    'pw_rigid': False,         # flag for performing piecewise-rigid motion correction (otherwise just rigid)
    'max_shifts': (35, 35),    # maximum allowed rigid shift
    'gSig_filt': (8, 8),       # size of high pass spatial filtering, used in 1p data
    'strides': (48, 48),       # start a new patch for pw-rigid motion correction every x pixels
    'overlaps': (24, 24),      # overlap between pathes (size of patch strides+overlaps)
    'max_deviation_rigid': 15,  # maximum deviation allowed for patch with respect to rigid shifts
    'border_nan': 'copy',      # replicate values along the boundaries
    'use_cuda': True,          # Set to True in order to use GPU
    'memory_fact': CONFIG['RAM']/16.0,          # How much memory to allocate. 1 works for 16Gb, so 0.8 showd be optimized for 12Gb.
    'niter_rig': 1,
    'splits_rig': 20,          # for parallelization split the movies in  num_splits chuncks across time
                               # if none all the splits are processed and the movie is saved
    'num_splits_to_process_rig': None} # intervals at which patches are laid out for motion correction  

for name in tqdm.tqdm(fnames):
    DoMotionCorrection(name, mc_dict)
    CleanMemmaps(name)

In [ ]:
ms_name = 'H02'
session_name = 'NOF_H02_0D'
mc_to_config = {'mc_params': mc_dict}

ms_config_path = get_mouse_config_path(ms_name)
session_config_path = get_session_config_path(session_name)

#update_config(mc_to_config, cpath=ms_config_path)
update_config(mc_to_config, cpath=session_config_path)

<h3>Module 2.5 (optional)</h3>
Pre-test of various values of <i>gSig</i> parameter, which is used in the Module 3 and corresponds to a typical radius of a neuron in pixels.<br/>You can play with this parameter but you can use the default value of gSig = 6 as well. <br/> Calculation may take a while, so be patient!

Code snippet for manual calculation of imax for corrupted images

In [ ]:
#%matplotlib ipympl
fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))
#fnames = askopenfilenames(title = 'Select files for corr image testing',
#                          initialdir = CONFIG['ROOT'],
#                          filetypes = [('TIFF files', '.tif')])

plot_gsig_range(fnames, maxframes=10000, min_gsig=3, max_gsig=6, step=5, dpi=300,
                    show_images=0, save_images=1)
plot_min_corr_and_pnr_range(fnames, maxframes=2000,
                            gsig_range=[3,4,5], pnr_range=[5,7,10],
                            mincorr_range=[0.85, 0.9, 0.95],
                            step=5, dpi=300,
                            show_images=0, save_images=1)

In [ ]:
'''session_name = ['RFC_F01_3D','RFC_F04_3D','RFC_F05_3D','RFC_F06_3D','RFC_F07_3D','RFC_F08_3D','RFC_F09_3D','RFC_F10_3D','RFC_F11_3D','RFC_F12_3D','RFC_F14_3D','RFC_F15_3D','RFC_F19_3D','RFC_F20_3D','RFC_F26_3D','RFC_F28_3D','RFC_F29_3D','RFC_F30_3D','RFC_F31_3D','RFC_F32_3D','RFC_F34_3D','RFC_F35_3D','RFC_F36_3D','RFC_F37_3D','RFC_F38_3D','RFC_F40_3D','RFC_F41_3D','RFC_F43_3D','RFC_F48_3D','RFC_F52_3D','RFC_F53_3D','RFC_F54_3D']
opt_gsig = [4,4,5,4,4,5,4,4,5,5,5,5,5,4,5,4,4,4,4,4,5,5,5,4,4,4,5,4,5,4,5,5]
min_corr = [0.95,0.9,0.9,0.9,0.85,0.95,0.85,0.85,0.9,0.9,0.9,0.9,0.95,0.9,0.95,0.9,0.9,0.9,0.9,0.9,0.95,0.95,0.95,0.9,0.9,0.9,0.9,0.9,0.95,0.9,0.9,0.9]
min_pnr = [7,7,7,10,5,7,5,7,5,7,7,5,7,7,15,10,7,10,7,10,10,10,10,7,7,7,10,10,10,5,10,10]
'''

session_name = ['BOF_H02_1T','BOF_H03_1T','BOF_H04_1T','BOF_H06_1T','BOF_H07_1T','BOF_H10_1T','BOF_H11_1T','BOF_H12_1T','BOF_H13_1T','BOF_H14_1T','BOF_H15_1T','BOF_H16_1T','BOF_H17_1T','BOF_H19_1T','BOF_H22_1T','BOF_H26_1T','BOF_H27_1T','BOF_H31_1T','BOF_H32_1T','BOF_H33_1T','BOF_H39_1T']
opt_gsig = [4,5,4,5,5,5,5,5,4,5,4,5,5,4,5,5,5,4,4,4,4]
min_corr = [0.85,0.9,0.9,0.9,0.95,0.9,0.9,0.9,0.85,0.9,0.85,0.9,0.87,0.88,0.9,0.9,0.95,0.9,0.9,0.9,0.9]
min_pnr = [5,6,7,7,7,8,7,8,6,7,6,6,6,8,7,8,9,8,8,8,8]

print(len(session_name), len(opt_gsig), len(min_corr), len(min_pnr))
for i, name in enumerate(session_name):
    gSiz = opt_gsig[i]*4+1
    cnmf_dict= {'fr': 30,                   # frame rate, frames per second (NOW RECALCULATED FOR EACH FILE FROM TIMESTAMP DATA)
                'decay_time': 1,            # typical duration of calcium transient 
                'method_init': 'corr_pnr',  # use this for 1 photon
                'K': None,                  # upper bound on number of components per patch, in general None
                'gSig': (opt_gsig[i], opt_gsig[i]),             # gaussian HALF-width of a 2D gaussian kernel (in pixels), which approximates a neuron
                'gSiz': (gSiz, gSiz),           # maximal radius of a neuron in pixels
                'merge_thr': 0.8,          # merging threshold, max correlation allowed
                'p': 1,                     # order of the autoregressive system
                'tsub': 1,                  # downsampling factor in time for initialization
                'ssub': 1,                  # downsampling factor in space for initialization
                'rf': 40,                   # half-size of the patches in pixels. e.g., if rf=40, patches are 80x80
                'stride': 25,               # amount of overlap between the patches in pixels(keep it at least large as gSiz, i.e 4 times the neuron size gSig) 
                'only_init': True,          # set it to True to run CNMF-E
                'nb': 0,                    # number of background components (rank) if positive, else exact ring model with following settings: nb= 0: Return background as b and W, gnb=-1: Return full rank background B, gnb<-1: Don't return background
                'nb_patch': 0,              # number of background components (rank) per patch if nb>0, else it is set automatically
                'method_deconvolution': 'oasis',       # could use 'cvxpy' alternatively
                'low_rank_background': None,           # None leaves background of each patch intact, True performs global low-rank approximation if gnb>0
                'update_background_components': True,  # sometimes setting to False improve the results
                'min_corr': min_corr[i],                        # min peak value from correlation image
                'min_pnr': min_pnr[i],                         # min peak to noise ratio from PNR image
                'normalize_init': False,               # just leave as is
                'center_psf': True,                    # leave as is for 1 photon
                'ssub_B': 2,                           # additional downsampling factor in space for background
                'ring_size_factor': 1.5,               # radius of ring is gSiz*ring_size_factor
                'del_duplicates': True,                # whether to remove duplicates from initialization
                'border_pix': 5,                       # number of pixels to not consider in the borders
                'min_SNR': 2.5,                          # adaptive way to set threshold on the transient size
                'rval_thr': 0.95,                      # threshold on space consistency           
                'use_cnn': False}                      # whether to use CNNs for event detection  

    session_config_path = get_session_config_path(session_name[i])
    cnmf_to_config = {'cnmf_params': cnmf_dict}
    update_config(cnmf_to_config, cpath=session_config_path)

<h2>Module 3</h2>
Batch cnmf.<br/>All _MC.tif files in the root folder are to be automatically processed with CaImAn routine [Giovanucci et al., 2019] with the parameters below. Main parameters are gSig and gSiz for cell augmentation, then min_SNR as traces quality threshold. At the end, _estimates.pickle files are to be produced in the root folder. 

In [ ]:
import time
#fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))
#OR, alternatively, you can use manual file selection:
fnames = askopenfilenames(title = 'Select files for batch cnmf', initialdir = CONFIG['ROOT'], filetypes = [('TIFF files', '.tif')])

for name in tqdm.tqdm(fnames):
    fps = get_fps_from_timestamps(name[:-4-6], default_fps=20, verbose=False)
    print('timestamps average FPS: ', fps)
    session_config_path = get_session_config_path(name[-20:-10])
    cnmf_config = read_config(name=session_config_path)
    cnmf_dict = cnmf_config['cnmf_params']
    cnmf_dict.update({'fr': fps/2})
    cnmf_dict.update({'tsub': 2})

    print(f"gsig: {cnmf_dict['gSig'][1]}, mincorr: {cnmf_dict['min_corr']}, minpnr: {cnmf_dict['min_pnr']}")
    out_name = name[:-4-6] + f'_estimates.pickle'
    print('estimate output name: ', out_name)
    DoCNMF(name,
           cnmf_dict,
           out_name=out_name,
           verbose=False,
           noise_ampl=1e-5)
    
    #CleanMemmaps(name)  
    temp_pathway = CONFIG['TEMP_PATHWAY']
    sub_name = '/'.join(name.split('/')[6:])
    memmap_name = f'{temp_pathway}/{sub_name}'
    
    err_cnt = 0
    while err_cnt < 100:
        try:
            CleanMemmaps(memmap_name)
            break
        except PermissionError:
            time.sleep(1)
            err_cnt += 1
    print('CleanMemmaps attemps:', err_cnt)

<h2>Module 4</h2>
User inspection of cnmf results.<br/>
Btw, at this stage, previously saved timestamps are to be merged with cnmf results.
By running the section below, you will be prompted to select estimtes file with cnmf results and then to interactively examine detected units (you can select both spatial and temporal components), you can select, merge and delete them, also you can seed new neurons (by PointDrawTool) for further re-run of CNMF with saved seeds. Finally, you can save (by pressing 'Save Results') spatial and temporal components as .tif and traces.csv files, respectively. Spatial components (aka filters) are to be stored in a separate folder (*_filters).

In [ ]:
import os
os.environ['BOKEH_ALLOW_WS_ORIGIN'] = 'localhost:8888'  # Укажите порт вашего Jupyter Notebook, например, 8888

In [2]:
import time

fname = askopenfilename(title = 'Select estimates file for examination',
                        initialdir = CONFIG['ROOT'],
                        filetypes = [('estimates files', '*.pickle')])

bkapp_kwargs = {
    'mode': 'capcan',          # operation mode, can be 'legacy'/'capcan'
    'start_frame': 0,          # start from this frame
    'end_frame': 90000,        # end at this frame
    'num_sessions': 1,         # for merged multi-session recordings
    'match_threshold': 1,      # threshold number of sessions with significant correlation, takes effect for merged multi-session recordings
    'include_event_based': 0,  # compute event-based metrics or not. Takes time (~0.2 s per neuron, a minute or two for typical exp size)
    'include_heavy': 0,        # compute heavy reconstruction-based metrics or not (may take about 5-10 minutes)
    'color_by_ml_probability': True,
    'ml_threshold': 0.7,
    'ml_model_path': 'ml/production_models/ebm_v5_inter20.pkl',  # Path to ML model for neuron classification
    'detect_corner_artifacts': True,   # Enable/disable corner artifact detection
    'corner_artifact_params': None,    # Custom params for corner detection (None = defaults)
    'correlation_method': 'pearson',   # 'pearson' or 'spearman' for correlation computation
    'downsampling': 5,         # take every 'ds' frame
    'fill_alpha': 0.8,         # selected neuron transparency
    'ns_alpha': 0.2,           # non-selected neuron transparency
    'line_width': 0.5,         # border width
    'cthr': 0.35,              # coutour_thr from caiman (% of signal inside a patch), affects patch size
    'sort_order': 'up',        # sorting order
    'corr_thr': 0.35,          # threshold for truncated corr matrix
    'line_alpha': 0.5,         # border transparency
    'trace_line_width': 1,     # trace line width
    'trace_alpha': 0.7,        # trace transparency
    'size': 500,               # left/right widget size
    'metrics_width': 200,      # central metrics widget width in pixels
    'button_width': 50,        # button width in pixels
    'compress_estimates': True,  # compress estimates before saving (reduces file size)
    'verbose': 0,
    'enable_gpu_backend': 1,
    'oh_shit': 0
}

ExamineCells(fname, default_fps=30, bkapp_kwargs=bkapp_kwargs)

/Users/nikita/PycharmProjects/bearmind/output/inspection_artifacts_NOF_H32_1D_
[]
Loaded cached reconstructions for 1225 neurons
Using pre-computed metrics from estimates.metrics_df (1088 neurons)
Elapsed time for all metrics: 0.01 s,0.0 s per neuron
Applied ML probability-based coloring (threshold=0.7)


In [7]:
from ae_launch import run_auto_inspection
'''
fname = askopenfilename(title = 'Select estimates file for examination',
                        initialdir = CONFIG['ROOT'],
                        filetypes = [('estimates files', '*estimates.pickle')])
'''
fname = 'data/NOF_H32_1D_gsig4_mincorr0.92_minpnr10_estimates.pickle'
result = run_auto_inspection(
    fname,
    fps=30,

    # --- Session naming ---
    session_name = None,

    # --- Metrics extraction parameters ---
    comps_to_select = None,
    cthr = 0.35,
    corr_thr = 0.6,
    num_sessions = 1,
    match_threshold = 3,
    sf = None,
    ef = None,
    ds = 1,
    include_event_based = True,
    include_heavy = True,
    detect_corner_artifacts = True,
    corner_artifact_params = None,
    event_method = 'threshold',  # Event detection method: 'threshold' or 'wavelet'
    correlation_method = 'pearson',  # Correlation method: 'pearson' or 'spearman'

    # --- Brain selection ---
    brain = 'ml',    # 'thresholds', 'ml', or 'hybrid' (thresholds first, then ML on survivors)
    ml_model_path = "ml/production_models/ebm_v5_inter20.pkl",
    ml_threshold = 0.7,

    # --- Decision parameters (threshold brain / hybrid brain) ---
    circ_thr = 4,
    maxedge_thr = 42,
    convex_thr = 42,
    pxlthr_area = 6.9,
    pxlthr_distance_boundary = 5,
    d_snr_thr = 10,
    t_rise_min = 0.10,
    caiman_r_score_min = 0.05,
    caiman_snr_min = 2.9,
    t_off_min = 1.5,

    # --- Enable/disable checks (for threshold/hybrid brain) ---
    use_circularity_check = True,
    use_area_check = True,
    use_max_edge_check = True,
    use_convexity_check = True,
    use_t_rise_check = True,
    use_caiman_r_score_check = True,
    use_caiman_snr_check = True,
    use_t_off_check = True,
    use_corr_check = True,
    
    # --- Tracking ---
    track_criteria_failures = True,

    # --- Artifact saving ---
    save_artifacts = True,
    artifacts_path = './output',
    save_estimates = True,
    save_matrices = True,
    save_corner_detection = True,
    compress_estimates = True,  # compress estimates before saving (reduces file size)

    # --- Verbosity ---
    verbose = True
)
 
est = result['estimates']  # Modified estimates object
print(f"Kept {len(est.idx_components)} neurons")

[run_auto_inspection] Session: NOF_H32_1D_gsig4_mincorr0.92_minpnr10
[run_auto_inspection] Loading estimates from: data/NOF_H32_1D_gsig4_mincorr0.92_minpnr10_estimates.pickle
data/NOF_H32_1D_
[]
[run_auto_inspection] Loaded 1231 components
[run_auto_inspection] Extracting metrics...
[1/4] Preparing traces for 1231 neurons...
[2/4] Computing correlation matrix (pearson)...
[3/4] Extracting spatial metrics...
Computing boundary distances...


1231it [00:00, 2641.32it/s]


[4/4] Computing threshold event-based metrics (this may take a while)...


KeyboardInterrupt: 

In [ ]:
fnames = askopenfilenames(title = 'Select files for batch cnmf',
                          initialdir = CONFIG['ROOT'],
                          filetypes = [('TIFF files', '.tif')])

#fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))

ManualSeeds(fnames[0], size=800, cnmf_dict=None)

Redo cnmf with manually added seeds (optional).<br/>
NB!! By running the cell below, you will rewrite existing estimates files!!<br/>
Then you can return to the section above and inspect the rewritten estimates.

In [ ]:
s_names = glob(os.path.join(CONFIG['ROOT'], '*seeds.pickle'))
#OR, alternatively, you can use manual file selection:
#s_names = askopenfilenames(title = 'Select seeds files for re-CNMFing', initialdir = CONFIG['ROOT'], filetypes = [('seeds files', '*seeds.pickle')])


for s_name in s_names:
    base_name = s_name.partition('_seeds')[0][:-4]
    
    e_names = glob(base_name + '_estimates.pickle')
    tif_names = glob(base_name + '.tif')
    ReDoCNMF(s_name, e_name=None, tif_name=tif_names[0], cnmf_dict=cnmf_dict)
    CleanMemmaps(base_name)

<h2>Module 5</h2>
Batch event detection. <br/>
INPUT: (timestamped) cnmf raw traces as *_traces.csv files<br/>
OUTPUT: detected events as *_spikes.csv files; pickles with events (cell-wise list of event-wise lists with dictionaries) and also, interactive .html plot with traces and events.

In [ ]:
fnames = glob(CONFIG['ROOT'] + '*traces.csv')
#OR, alternatively, you can use manual file selection:
#fnames = askopenfilenames(title = 'Select traces for event detection', initialdir = CONFIG['ROOT'], filetypes = [('traces files', '*traces.csv')])

sd_dict = {'thr': 4,        #threshold for peaks in Median Absolute Deviations (MADs)                   
           'sigma' : 7,     #smoothing parameter for peak detection, frames
           'est_ton' : 0.5, #estimated event rising time, s
           'est_toff' : 2,  #estimated event decay time, s
           'draw_details': True} #whether to draw smoothed traces, peaks, pits and fits 

for name in fnames:
    FitEvents(name, opts = sd_dict)


Also, just in case, you may draw existed pairs of traces and spikes right here:

In [ ]:
fnames = glob(CONFIG['ROOT'] + '*traces.csv')
for name in fnames:
    DrawSpEvents(name, name.replace('traces','spikes'))


<h1> Wavelet event detection</h1>

In [ ]:
import scipy
print(scipy.__version__)


In [ ]:
def read_traces(fname):
    trdata = pd.read_csv(fname)
    time = trdata['time_s'].values
    traces = np.array(trdata)[:,1:].T
    return traces, time

#fnames = glob(CONFIG['ROOT'] + '*traces.csv')
fnames = askopenfilenames(title = 'Select traces for event detection', initialdir = CONFIG['ROOT'], filetypes = [('traces files', '*traces.csv')])

wvt_param_dict = {'fps': 30,        # fps, frames                   
                  'sigma' : 8,      # smoothing parameter for peak detection, frames
                  'beta' : 2,       # Generalized Morse Wavelet parameter, FIXED
                  'gamma' : 3,      # Generalized Morse Wavelet parameter, FIXED
                  'eps': 10,         # spacing beween consecutive events, frames
                  'manual_scales': np.logspace(2.5,5.5,50, base=2),

                  # ridge filtering params
                  'scale_length_thr': 40,  # min number of scales where ridge is present thr, higher = less events. max=len(manual_scales)
                  'max_scale_thr': 7,      # index of a scale with max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
                  'max_ampl_thr': 0.05,    # max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
                  'max_dur_thr': 100,      # max event duration thr, higher = more events (but probably strange ones)
}


for fname in fnames:
    traces, time = read_traces(fname)
    st_evinds, end_evinds, all_ridges = extract_wvt_events(traces, wvt_param_dict)
    events_to_csv(time, st_evinds, end_evinds, fname)

Recompute with different filtering params:

In [ ]:
wvt_param_dict['scale_length_thr'] = 40,  # min number of scales where ridge is present thr, higher = less events. max=len(manual_scales)
wvt_param_dict['max_scale_thr'] = 7,      # index of a scale with max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
wvt_param_dict['max_ampl_thr'] = 0.05,    # max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
wvt_param_dict['max_dur_thr'] = 200,      # max event duration thr, higher = more events (but probably strange ones)

events = []
for i in range(traces.shape[0]):
    st_evinds, end_evinds = get_events_from_ridges(all_ridges[i],
                                                   scale_length_thr=40,
                                                   max_scale_thr=7,
                                                   max_ampl_thr=0.05,
                                                   max_dur_thr=200)

    events.append(end_evinds)

In [ ]:
st = 0
end = 10000
neuron_ind = 10

sig = gaussian_filter1d(traces[neuron_ind], sigma=wvt_param_dict['sigma'])
sig = traces[neuron_ind]

fig, ax = plt.subplots(figsize=(10,8))
ax.set_xlim(st, end)
ax.plot(np.arange(st, end), sig[st:end], c='b')


#for ev in end_evinds[neuron_ind]:
#    ax.axvline(ev, c='r', alpha=0.5)

for ev in events[neuron_ind]:
    ax.axvline(ev, c='r', alpha=0.5)